# Importation des bibliothèques
Dans cette cellule, nous importons toutes les bibliothèques nécessaires pour notre projet.

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import kagglehub

2025-01-06 14:28:26.471451: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/Caskroom/miniforge/base/envs/tensorflow_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Chemins des données
Dans cette cellule, nous définissons les chemins pour les ensembles de données réels et faux.

In [2]:
REAL_FACES_DIR = kagglehub.dataset_download("tunguz/70000-real-faces-1")
FAKE_FACES_DIR = kagglehub.dataset_download("tunguz/1-million-fake-faces")

Resuming download from 3569352704 bytes (16922294702 bytes left)...
Resuming download from https://www.kaggle.com/api/v1/datasets/download/tunguz/70000-real-faces-1?dataset_version_number=10 (3569352704/20491647406) bytes left.


 47%|████▋     | 8.90G/19.1G [04:00<06:01, 30.3MB/s]  

# Charger les images et leurs labels
Cette fonction charge les images à partir des répertoires spécifiés et attribue des labels.

In [ ]:
def load_images(directory, label):
    images = []
    labels = []
    for root, _, files in os.walk(directory):  # Traverse directories recursively
        for filename in files:
            if filename.lower().endswith(('.png', '.jpg', '.jpeg')):  # Check for image extensions
                filepath = os.path.join(root, filename)
                try:
                    image = tf.keras.preprocessing.image.load_img(filepath, target_size=(128, 128))
                    image = tf.keras.preprocessing.image.img_to_array(image)
                    images.append(image)
                    labels.append(label)
                except Exception as e:
                    print(f"Error loading image {filepath}: {e}")
    return images, labels

# Charger les données
Nous chargeons les images réelles et fausses en utilisant la fonction définie précédemment.

In [ ]:
real_images, real_labels = load_images(REAL_FACES_DIR, 0)
fake_images, fake_labels = load_images(FAKE_FACES_DIR, 1)

# Combiner et normaliser
Nous combinons les images réelles et fausses, puis les normalisons.

In [ ]:
images = np.array(real_images + fake_images, dtype="float32") / 255.0
labels = np.array(real_labels + fake_labels)

# Diviser les données
Nous divisons les données en ensembles d'entraînement, de validation et de test.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(images, labels, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Augmentation des données
Nous définissons les paramètres d'augmentation des données pour améliorer la robustesse du modèle.

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest"
)
train_generator = datagen.flow(X_train, y_train, batch_size=32)

# Construire le modèle
Nous construisons le modèle de réseau de neurones en utilisant des couches convolutionnelles.

In [ ]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Définir les callbacks
Nous définissons les callbacks pour l'entraînement, y compris l'arrêt précoce et le point de contrôle du modèle.

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint('best_model.h5', save_best_only=True)
]

# Entraîner le modèle
Nous entraînons le modèle en utilisant les données d'entraînement et de validation.

In [ ]:
history = model.fit(
    train_generator,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32,
    callbacks=callbacks
)

# Évaluation et sauvegarde
Nous évaluons le modèle sur les données de test et sauvegardons le modèle entraîné.

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
model.save("face_classifier_model.h5")